In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [2]:
#Feature
def generate_batch(batch_size, S, n):
    input1 = torch.rand(batch_size, n)
    input2 = torch.rand(batch_size, n)
    dataset = input1 * torch.gt(input2, S).float()
    return dataset

In [3]:
# Neural Network Model
class NeuralNetwork(nn.Module):
    def __init__(self, n, m, activation):
        super(NeuralNetwork, self).__init__()
        self.n = n
        self.m = m
        self.weight = nn.init.xavier_normal_(torch.rand(m, n))
        self.bias = torch.zeros(n)
        self.activation = activation
        self.weight = nn.Parameter(self.weight)
        self.bias = nn.Parameter(self.bias)


    def forward(self, x):
        h = torch.matmul(x, self.weight.t())
        x_prime = self.activation(torch.matmul(h, self.weight) + self.bias)
        return x_prime
    

In [4]:
n = 5 # number of input features
m = 2 # number of hidden units
importance = torch.arange(0, n).float()  # importance of each feature
importance = torch.pow(0.7, importance)  # apply a power transformation to the importance values
def loss_fn (x, x_prime, importance):
    return torch.sum(importance * (x - x_prime) ** 2, dim=1).mean()
# m < n to ensure that the model is underparameterized
model = NeuralNetwork(n, m, nn.ReLU())
dataset = generate_batch(3, 0, n)

In [22]:
# Training loop model function
def train_model(n, batch_size, S, model, importance, epochs=7000, learning_rate=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)
    for epoch in range(epochs):
        dataset = generate_batch(batch_size, S, n)
        optimizer.zero_grad()
        x_prime = model(dataset)
        loss = loss_fn(dataset, x_prime, importance)
        loss.backward()
        optimizer.step()
        if epoch == epochs - 1:
            print(f"Weights for S = {S}: {model.weight.detach().numpy()}")

In [26]:
# Two features weight vector test to show superposition of features in the hidden layer
n = 2 # number of input features
m = 1 # number of hidden units
batch_size = 1000
importance = torch.ones(n).float()  # importance of each feature
S = 0.9 # threshold for generating the dataset
model_1 = NeuralNetwork(n, m, nn.ReLU())
train_model(n, batch_size, S, model_1, importance)

Weights for S = 0.9: [[-1.0062556  1.0068637]]


In [24]:
# Larger feature weight vector test to show superposition of features in the hidden layer
n = 20 # number of input features
m = 5 # number of hidden units
importance = torch.arange(0, n).float()  # importance of each feature
importance = torch.pow(0.7, importance)  # apply a power transformation to the importance values
S = [0, 0.7, 0.9, 0.97, 0.99] # thresholds for generating the dataset
model_relu = NeuralNetwork(n, m, nn.ReLU())
for S in S:
    train_model(n, batch_size, S, model_relu, importance)

Weights for S = 0: [[ 7.54426599e-01 -3.61616492e-01  1.47343889e-01 -3.65674317e-01
   3.80502343e-01  7.48772640e-04 -3.98184219e-03 -1.16062118e-04
   7.89954094e-04  4.17346600e-04  9.51415917e-04  4.78358706e-04
   9.90146800e-05  1.39755248e-05  3.30415409e-04 -9.23662592e-05
   2.98829545e-04  2.98563245e-04  1.09443296e-04  2.04218042e-04]
 [ 2.38316417e-01 -5.34281313e-01 -4.00961369e-01  6.84946835e-01
  -1.66688770e-01 -3.59799503e-03  1.99635886e-03 -1.05945300e-03
   1.49612140e-03 -8.24257382e-04  6.25282628e-05 -4.01422381e-04
   1.68156286e-04  4.23731362e-05  4.05117898e-04  3.29065486e-04
  -2.21419978e-05  5.12552906e-05  2.17727749e-04  2.56639207e-04]
 [ 4.35793340e-01  6.29463434e-01  2.97930330e-01  5.50224245e-01
   1.48084432e-01 -1.24941731e-03 -3.84902640e-04  3.00486037e-03
  -1.75266713e-03  5.04838419e-04  2.93029763e-04  1.60926220e-03
  -1.57448254e-03 -1.87207173e-04 -3.08833405e-05 -7.54459179e-04
   5.80989072e-05 -5.06149736e-05 -2.34026375e-04 -2.34

In [ ]:
model_linear = NeuralNetwork(n, m, nn.Identity())
for S in S:
    train_model(n, batch_size, S, model_linear, importance)

Weights: tensor([[ 5.0060e-01, -6.9244e-01, -1.7976e-01,  1.5312e-01, -4.6282e-01,
          2.7030e-03, -1.6389e-03,  2.4074e-03,  1.7573e-05,  5.7312e-04,
          3.5077e-04, -2.3773e-04, -1.7368e-05, -7.0238e-05, -6.5619e-04,
          1.6352e-04,  5.9919e-06, -1.8244e-04,  1.7460e-05,  6.4272e-05],
        [-6.8365e-01, -1.0975e-01, -5.6733e-01,  3.8305e-01, -2.2820e-01,
          3.2171e-03, -1.5594e-03,  2.6018e-04,  7.6424e-05,  4.7577e-05,
         -2.7486e-04,  2.8209e-04,  1.2897e-04,  1.8572e-04, -4.2831e-04,
         -1.5775e-04, -1.0926e-04,  4.4185e-06,  2.1456e-05,  1.1665e-04],
        [-4.1438e-01, -9.3274e-02,  6.2266e-01, -2.1407e-01, -6.2123e-01,
          3.8928e-03, -2.3248e-03,  4.4927e-03,  1.9457e-03,  9.2144e-05,
         -3.7591e-04, -5.1592e-04, -4.4616e-04, -1.4060e-04, -5.4986e-05,
          7.7107e-05,  1.5142e-06, -1.8191e-04, -5.8625e-05,  3.6104e-05],
        [ 8.2819e-03,  1.0783e-01, -4.9660e-01, -8.2889e-01, -2.3395e-01,
          3.0656e-03,  4.1